# Friedrichs, Poincaré, and trace constants

The inequalities in Chapter 3 of the manuscript have constants that depend on the domain and on the additional condition imposed on the functions. We investigate these dependencies numerically.

Our starting point is the fixed square $\Omega=(0,1)^2$. We compare a fixed Dirichlet boundary, a global mean condition, shrinking boundary and volume measurements, and a single fixed finite element vertex. An optional final experiment changes the domain itself.

**Packages.** Besides NGSolve/Netgen, this notebook uses **NumPy, SciPy and Matplotlib**. SciPy provides the sparse eigenvalue solver. If needed, install the additional packages with `python -m pip install numpy scipy matplotlib` in your Jupyter environment.

The eigenvalue approach is adapted from J. Schöberl's [Experiments with norms](https://jschoeberl.github.io/iFEM/sobolevspaces/experiments.html).

In [ ]:
from ngsolve import *
from netgen.occ import Rectangle, Glue, OCCGeometry, unit_square
from ngsolve.webgui import Draw
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix, csr_matrix
from scipy.sparse.linalg import eigsh

## 1. Why does the gradient need an additional condition?

For $u\equiv1$, we have $\nabla u=0$ but $\|u\|_{L^2(\Omega)}>0$. Thus the gradient seminorm alone cannot control the function on $H^1(\Omega)$.

Two results give ways to remove the constant functions:

- **Friedrichs inequality:** for a boundary part $\Gamma_D$ of positive length, $|\Gamma_D|>0$,
  
  $$
  \begin{align*}
  \|u\|_{L^2(\Omega)}\le C_F\|\nabla u\|_{L^2(\Omega)} \qquad \forall u\in H_D^1(\Omega), \qquad H_D^1(\Omega)=\{u\in H^1(\Omega):\gamma u=0\text{ on }\Gamma_D\}.
  \end{align*}
  $$

- **Classical Poincaré inequality:** with $\bar u=|\Omega|^{-1}\int_\Omega u\,dx$,
  
  $$
  \begin{align*}
  \|u-\bar u\|_{L^2(\Omega)}\le C_P\|\nabla u\|_{L^2(\Omega)} \qquad \forall u\in H^1(\Omega).
  \end{align*}
  $$

### Computing the best constant on a finite element space

For an inequality $b(u,u)\le C^2 a(u,u)$, the sharp discrete constant is

$$
\begin{align*}
C_h^2=\max_{0\ne u_h\in V_h}\frac{b(u_h,u_h)}{a(u_h,u_h)}=\frac1{\lambda_{\min}},\qquad A\mathbf u=\lambda B\mathbf u.
\end{align*}
$$

Here $a$ must be positive on the space under consideration. For Poincaré, we exclude the constant Neumann eigenfunction: if $0=\lambda_1<\lambda_2\le\cdots$, then $C_{P,h}=1/\sqrt{\lambda_2}$.

The following helpers are provided. They assemble the $L^2$ mass matrix $M$, the gradient matrix $K$ and the full $H^1$ matrix $H=M+K$. The eigenvalue helper restricts the matrices to the free degrees of freedom and returns $L^2$-normalized finite element eigenfunctions. It also supports the largest eigenvalue, needed for the trace experiment.

Whenever the continuous inequality holds, the finite element spaces restrict the maximization, so the resulting constants are **lower approximations** to its sharp constant.

In [ ]:
def sparse_matrix(matrix):
    rows, columns, values = matrix.COO()
    return coo_matrix((values, (rows, columns)),
                      shape=(matrix.height, matrix.width)).tocsr()


def norm_matrices(fes):
    u, v = fes.TnT()
    M = sparse_matrix(BilinearForm(u*v*dx).Assemble().mat)
    K = sparse_matrix(BilinearForm(grad(u)*grad(v)*dx).Assemble().mat)
    return M, K, M + K


def eigenpairs(A, B, fes, num=1, freedofs=None, largest=False):
    if freedofs is None:
        freedofs = fes.FreeDofs()
    free = np.flatnonzero(list(freedofs))
    A_free = A[free][:, free]
    B_free = B[free][:, free]
    options = {"which": "LA"} if largest else {"sigma": -1.0, "which": "LM"}
    values, vectors = eigsh(
        A_free, k=num, M=B_free, tol=1e-10,
        v0=np.random.default_rng(1).standard_normal(len(free)), **options)
    indices = np.argsort(values)
    modes = []
    for j in indices:
        mode = GridFunction(fes)
        mode.vec.FV().NumPy()[free] = vectors[:, j]
        mode.vec.data /= sqrt(Integrate(mode**2, fes.mesh))
        if Integrate(mode, fes.mesh) < -1e-10:
            mode.vec.data *= -1
        modes.append(mode)
    return values[indices], modes

### Fixed conditions on the unit square

First take $\Gamma_D=\{0\}\times(0,1)$ for Friedrichs, and no Dirichlet condition for Poincaré, $\Gamma_D=\emptyset$. The remaining boundary conditions in the Friedrichs eigenproblem are homogeneous Neumann conditions.

For comparison, the exact sharp constants in these two cases are

$$
\begin{align*}
C_F=\frac2\pi,\qquad C_P=\frac1\pi.
\end{align*}
$$

The first follows from $u(x,y)=\sin(\pi x/2)$. For Poincaré, the first positive Neumann eigenvalue is $\pi^2$.

In [ ]:
maxhs = [0.25, 0.125, 0.0625, 0.0625/2]
C_F_fixed, C_P_fixed = [], []

for maxh in maxhs:
    mesh = Mesh(unit_square.GenerateMesh(maxh=maxh))

    fes_D = H1(mesh, order=1, dirichlet="left")
    M, K, H = norm_matrices(fes_D)
    values_D, modes_D = eigenpairs(K, M, fes_D)
    C_F_fixed.append(1/np.sqrt(values_D[0]))

    fes_N = H1(mesh, order=1)
    M, K, H = norm_matrices(fes_N)
    values_N, modes_N = eigenpairs(K, M, fes_N, num=3)
    C_P_fixed.append(1/np.sqrt(values_N[1]))

    print(f"maxh={maxh:.4f}: C_F,h={C_F_fixed[-1]:.6f}, "
          f"C_P,h={C_P_fixed[-1]:.6f}")

print("2/pi = ", 2/np.pi, "1/pi =", 1/np.pi)
print("Neumann eigenvalues on the finest mesh:", values_N)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), constrained_layout=True)
for ax, constants, exact, name in zip(
        axes, [C_F_fixed, C_P_fixed], [2/np.pi, 1/np.pi], ["F", "P"]):
    ax.semilogx(1/np.array(maxhs), constants, "o-", label="finite elements")
    ax.axhline(exact, color="black", linestyle="--", label="exact constant")
    ax.set(xlabel="1 / maxh", ylabel=rf"$C_{{{name},h}}$",
           title="Fixed Dirichlet side" if name == "F" else "Global zero mean")
    ax.grid(True)
    ax.legend()
plt.show()

Draw(modes_D[0], name="Friedrichs: fixed left side", deformation=True);
Draw(modes_N[1], name="Poincare: first nonconstant Neumann mode", deformation=True);

The constants approach finite limits.

For Friedrichs with the fixed left Dirichlet side, the first eigenvalue is $\lambda_1=\pi^2/4$. Up to its sign, the corresponding $L^2$-normalized eigenfunction is

$$
\begin{align*}
u(x,y)=\sqrt{2}\sin(\pi x/2).
\end{align*}
$$

It is independent of $y$, vanishes at $x=0$, and is positive and increasing for $0<x<1$. At $x=1$ its slope is zero, as required by the homogeneous Neumann condition. Its graph is one quarter of a sine wave.

The Poincaré eigenfunction has zero mean and changes sign. The eigenvalue $\pi^2$ has multiplicity two on the square, so its computed eigenfunction can be any linear combination of $\cos(\pi x)$ and $\cos(\pi y)$.

## 2. A shrinking Dirichlet boundary

Keep $\Omega=(0,1)^2$ fixed and take

$$
\begin{align*}
\Gamma_{D,\varepsilon}=S_\varepsilon=(0,\varepsilon)\times\{0\}.
\end{align*}
$$

For each $\varepsilon>0$, the Friedrichs inequality applies. Its constant may depend on $\Gamma_{D,\varepsilon}$.

The geometry below splits off $V_\varepsilon=(0,\varepsilon)^2$. This supplies both a fitted volume region `V` and a boundary part `S`. The union is still the unit square. Small elements resolve the measurement region and the end of the Dirichlet segment.

In [ ]:
def measurement_mesh(epsilon, maxh=0.1, local_divisions=8):
    small = Rectangle(epsilon, epsilon).Face()
    small.faces.name = "V"
    small.faces.maxh = min(maxh, epsilon/local_divisions)
    small.edges.maxh = min(maxh, epsilon/local_divisions)
    remainder = Rectangle(1, 1).Face() - small
    remainder.faces.name = "remainder"
    shape = Glue([remainder, small])

    for edge in shape.edges:
        cx, cy, _ = edge.center
        if abs(cy) < 1e-10:
            edge.name = "S" if cx < epsilon else "bottom"
        elif abs(cx) < 1e-10:
            edge.name = "left"
        elif abs(cx-1) < 1e-10:
            edge.name = "right"
        elif abs(cy-1) < 1e-10:
            edge.name = "top"
        else:
            edge.name = "interface"
    return Mesh(OCCGeometry(shape, dim=2).GenerateMesh(maxh=maxh))


mesh = measurement_mesh(0.125)
Draw(mesh, name="Unit square with fitted V and S");

In [ ]:
epsilons = np.array([0.5, 0.25, 0.125, 0.0625, 0.03125, 0.015625])
C_F_shrinking = []
shrinking_modes = []

# Refine both the global mesh and the local resolution of the small region
for maxh, local_divisions in [(0.2, 4), (0.1, 8)]:
    constants = []
    for epsilon in epsilons:
        mesh = measurement_mesh(epsilon, maxh, local_divisions)
        fes = H1(mesh, order=1, dirichlet="S")
        M, K, H = norm_matrices(fes)
        values, modes = eigenpairs(K, M, fes)
        constants.append(1/np.sqrt(values[0]))
        if local_divisions == 8 and epsilon in (epsilons[0], epsilons[-1]):
            shrinking_modes.append(modes[0])
    C_F_shrinking.append(constants)

for epsilon, coarse, fine in zip(epsilons, *C_F_shrinking):
    print(f"epsilon={epsilon:.6f}: coarse={coarse:.6f}, fine={fine:.6f}")

In [ ]:
plt.figure(figsize=(6, 3.5))
for constants, label in zip(C_F_shrinking, ["coarser mesh", "finer mesh"]):
    plt.semilogx(1/epsilons, constants, "o-", label=label)
plt.xlabel(r"$1/\varepsilon$")
plt.ylabel(r"$C_{F,h}(\varepsilon)$")
plt.title("Shrinking Dirichlet segment on a fixed square")
plt.grid(True)
plt.legend()
plt.show()

Draw(shrinking_modes[0], name="Dirichlet segment: epsilon=1/2", deformation=True);
Draw(shrinking_modes[1], name="Dirichlet segment: epsilon=1/64", deformation=True);

The function is fixed to zero only on a short segment. Away from that segment, it can be close to a nonzero constant while spending little gradient energy. Thus the Friedrichs constants become large as the segment collapses to a point.

For each fixed $\varepsilon>0$, mesh refinement approximates a problem with a finite continuous constant. The limit $\varepsilon\to0$ changes that problem. Comparing the two mesh resolutions helps distinguish these effects.

## 3. Shrinking measurements of constants

The Sobolev norm-equivalence theorem applies whenever a bounded linear map $T$ detects the constant functions. For example,

$$
\begin{align*}
Tu=\int_{V_\varepsilon}u\,dx \quad\text{or}\quad Tu=\int_{S_\varepsilon}\gamma u\,ds, \qquad |V_\varepsilon|=\varepsilon^2,\quad |S_\varepsilon|=\varepsilon.
\end{align*}
$$

For every fixed $\varepsilon>0$, there is a finite constant in

$$
\begin{align*}
\|u\|_{H^1(\Omega)}\le C_T(\varepsilon)\left(\|\nabla u\|_{L^2(\Omega)}^2+|Tu|^2\right)^{1/2}.
\end{align*}
$$

We also try the normalized measurements $|V_\varepsilon|^{-1}\int_{V_\varepsilon}u\,dx$ and $|S_\varepsilon|^{-1}\int_{S_\varepsilon}\gamma u\,ds$.

Testing the unnormalized estimates with $u\equiv1$, for which $\|u\|_{H^1(\Omega)}=1$, gives

$$
\begin{align*}
C_T(\varepsilon)\ge
\begin{cases}
\varepsilon^{-2}, & \text{for the volume integral},\\
\varepsilon^{-1}, & \text{for the boundary integral}.
\end{cases}
\end{align*}
$$

These are the black reference curves in the plot below. For the normalized measurements, $T1=1$, so this test only gives $C_T(\varepsilon)\ge1$.

If $\ell_i=T\phi_i$, then $(Tu_h)^2=\mathbf u^\top\ell\ell^\top\mathbf u$. The energy matrix is therefore $A=K+\ell\ell^\top$, and the target matrix for the **full** $H^1$ norm is $B=H=M+K$.

In [ ]:
def measurement_constant(fes, K, H, ell):
    column = csr_matrix(ell[:, None])
    A = K + column @ column.T
    values, _ = eigenpairs(A, H, fes)
    return 1/np.sqrt(values[0])


C_V, C_S, C_V_mean, C_S_mean = [], [], [], []
for epsilon in epsilons:
    mesh = measurement_mesh(epsilon)
    fes = H1(mesh, order=1)
    M, K, H = norm_matrices(fes)
    u, v = fes.TnT()
    volume = Integrate(1, mesh, definedon=mesh.Materials("V"))
    length = Integrate(CF(1)*ds("S"), mesh)
    ell_V = LinearForm(v*dx("V")).Assemble().vec.FV().NumPy().copy()
    ell_S = LinearForm(v*ds("S")).Assemble().vec.FV().NumPy().copy()

    C_V.append(measurement_constant(fes, K, H, ell_V))
    C_S.append(measurement_constant(fes, K, H, ell_S))
    C_V_mean.append(measurement_constant(fes, K, H, ell_V/volume))
    C_S_mean.append(measurement_constant(fes, K, H, ell_S/length))
    print(f"epsilon={epsilon:.6f}: volume integral={C_V[-1]:.3f}, "
          f"boundary integral={C_S[-1]:.3f}, "
          f"volume mean={C_V_mean[-1]:.3f}, boundary mean={C_S_mean[-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
axes[0].loglog(1/epsilons, C_V, "o-", label="volume integral")
axes[0].loglog(1/epsilons, C_S, "s-", label="boundary integral")
axes[0].loglog(1/epsilons, epsilons**(-2), "k--", label=r"$1/\varepsilon^2$")
axes[0].loglog(1/epsilons, epsilons**(-1), "k:", label=r"$1/\varepsilon$")
axes[0].set_title("Unnormalized measurements")
axes[1].semilogx(1/epsilons, C_V_mean, "o-", label="volume mean")
axes[1].semilogx(1/epsilons, C_S_mean, "s-", label="boundary mean")
axes[1].set_title("Normalized measurements")
for ax in axes:
    ax.set(xlabel=r"$1/\varepsilon$", ylabel=r"$C_{T,h}(\varepsilon)$")
    ax.grid(True)
    ax.legend()
plt.show()

Normalization removes the immediate loss seen by testing a constant function. It does not give a uniform estimate as the measurement region collapses to a point in two dimensions: functions can be small near the measurement and nearly constant elsewhere.

The domain $\Omega$ has not changed in these experiments. Only the region used by $T$ has shrunk. The theorem guarantees norm equivalence for each fixed region of positive measure, but it does not assert one common constant for all these regions.

## 4. Fixing a single finite element vertex

On a fixed mesh, continuous piecewise linear functions have point values. We can impose $u_h(0,0)=0$ by removing the degree of freedom at that vertex, and compute

$$
\begin{align*}
C_{\mathrm{point},h}=\max_{\substack{0\ne u_h\in V_h\\u_h(0,0)=0}}\frac{\|u_h\|_{L^2(\Omega)}}{\|\nabla u_h\|_{L^2(\Omega)}}.
\end{align*}
$$

We keep the point $(0,0)$ fixed and refine the mesh. There are no Dirichlet edges in this experiment.

In [ ]:
point_maxhs = [0.5, 0.25, 0.125, 0.0625, 0.03125, 0.015625, 0.015625/2]
C_point = []
point_modes = []

for maxh in point_maxhs:
    mesh = Mesh(unit_square.GenerateMesh(maxh=maxh))
    fes = H1(mesh, order=1)
    M, K, H = norm_matrices(fes)

    # Identify the corner geometrically, rather than assuming a dof number
    corner = next(vertex for vertex in mesh.vertices
                  if abs(vertex.point[0]) + abs(vertex.point[1]) < 1e-12)
    corner_dof = fes.GetDofNrs(NodeId(VERTEX, corner.nr))[0]
    free = BitArray(fes.FreeDofs())
    free[corner_dof] = False

    values, modes = eigenpairs(K, M, fes, freedofs=free)
    C_point.append(1/np.sqrt(values[0]))
    point_modes.append(modes[0])
    print(f"maxh={maxh:.6f}: C_point,h={C_point[-1]:.6f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), constrained_layout=True)
axes[0].semilogx(1/np.array(point_maxhs), C_point, "o-", label="one fixed vertex")
axes[0].axhline(1/np.pi, color="black", linestyle="--",
                label=r"global zero mean: $C_P=1/\pi$")
axes[0].set(xlabel="1 / maxh", ylabel="constant in the L2 estimate",
            title="A point condition under mesh refinement")
axes[1].plot(np.log(1/np.array(point_maxhs)), np.array(C_point)**2,
             "o-", label="one fixed vertex")
axes[1].set(xlabel="log(1 / maxh)", ylabel=r"$C_{\mathrm{point},h}^2$",
            title="Squared constant versus log(1 / maxh)")
for ax in axes:
    ax.grid(True)
    ax.legend()
plt.show()

Draw(point_modes[-1], name="Finest mesh: one fixed corner vertex", deformation=True);

For each fixed mesh, the point condition removes the constant kernel, and the discrete constant is finite. Along the refinement sequence, however, the constants grow. The nearly straight curve in the second panel indicates approximately logarithmic growth of $C_{\mathrm{point},h}^2$ over the tested mesh sizes. In two dimensions, point evaluation is bounded on $H^s(\Omega)$ for $s>1$, but not on $H^1(\Omega)$. Therefore, it cannot serve as $T$ in the norm-equivalence theorem on $H^1(\Omega)$.

The computed function is zero at the pinned vertex but can remain almost constant over most of the square. The transition becomes concentrated near the vertex.

In one dimension, point evaluation on $H^1$ is continuous, and fixing an endpoint does give a mesh-independent Friedrichs inequality. Thus the spatial dimension matters here.

## 5. Optional: the trace constant on a thin rectangle

The trace inequality reads

$$
\begin{align*}
\|\gamma u\|_{L^2(\partial\Omega)} \le C_{\mathrm{tr}}\|u\|_{H^1(\Omega)}.
\end{align*}
$$

Take $\Omega_\delta=(0,1)\times(0,\delta)$ and compute the constant for the **full boundary**

$$
\begin{align*}
C_{\mathrm{tr},h}^2=\max_{0\ne u_h\in V_h}\frac{\|\gamma u_h\|_{L^2(\partial\Omega_\delta)}^2}{\|u_h\|_{H^1(\Omega_\delta)}^2}.
\end{align*}
$$

This is the largest eigenvalue of the boundary mass matrix relative to $H$.

Note, that the constant function gives

$$
\begin{align*}
C_{\mathrm{tr}}\ge\sqrt{\frac{|\partial\Omega_\delta|}{|\Omega_\delta|}}=\sqrt{\frac{2(1+\delta)}{\delta}}.
\end{align*}
$$

Here the whole domain changes. This differs from shrinking a measurement region inside a fixed domain.

In [ ]:
deltas = np.array([1.0, 0.5, 0.25, 0.125, 0.0625, 0.0625/2])
C_trace = []
for delta in deltas:
    shape = Rectangle(1, delta).Face()
    # Resolve the thickness of the rectangle in each experiment
    mesh = Mesh(OCCGeometry(shape, dim=2).GenerateMesh(maxh=min(0.15, delta/3)))
    fes = H1(mesh, order=1)
    M, K, H = norm_matrices(fes)
    u, v = fes.TnT()
    boundary_mass = sparse_matrix(BilinearForm(u*v*ds).Assemble().mat)
    values, modes = eigenpairs(boundary_mass, H, fes, largest=True)
    C_trace.append(np.sqrt(values[-1]))

plt.figure(figsize=(6, 3.5))
plt.loglog(1/deltas, C_trace, "o-", label="finite elements")
plt.loglog(1/deltas, np.sqrt(2*(1+deltas)/deltas), "k--",
           label="lower bound from the constant function")
plt.xlabel(r"$1/\delta$")
plt.ylabel(r"$C_{\mathrm{tr},h}$")
plt.title("Full boundary trace on a thin rectangle")
plt.grid(True)
plt.legend()
plt.show()

A small domain volume alone does not imply large Friedrichs or Poincaré constants. Under uniform scaling $\Omega_t=t\Omega$, their sharp constants scale by $t$. For Friedrichs inequality, the Dirichlet part must also be scaled as $\Gamma_{D,t}=t\Gamma_D$. On these thin rectangles, $C_P$ remains bounded while $C_{\mathrm{tr}}$ grows. The Poincaré bound is consistent with the convex-domain estimate $C_P\le\operatorname{diam}(\Omega_\delta)/\pi$.

## Takeaways

- For a fixed positive-size Dirichlet part or a global mean condition, mesh refinement gives stable constants.
- Shrinking the region that controls the constant kernel can make the equivalence constants arbitrarily large.
- Integral measurements and normalized means have different scaling.
- A single finite element vertex gives a finite constant on each mesh, but no uniform bound under refinement in two dimensions.
- The constants depend on the inequality, its normalization, the geometry and the imposed condition.

[Lecture notebooks](../README.md) · [Course overview](../../index.ipynb)